# Apache Spark — Airbnb Dublin Analysis

Benchmark of data processing operations on the **Dublin Airbnb dataset** using PySpark.

| File | Description |
|------|-------------|
| `listings.csv` | ~7,000 listings — price, location, ratings |
| `calendar.csv.gz` | ~2.5M rows — daily availability and pricing |

**Pipeline:** Init Spark → Load → Clean → Aggregate calendar → Join → Group by neighbourhood × room type

> **Note:** Each timing cell calls `.persist()` + `.count()` to force Spark to materialise the DataFrame before recording the time — this mirrors Pandas' eager execution.

To run with the **large dataset** (5× replicated, calendar read as parallel part files):
```
docker compose run --rm \
  -e LISTINGS_FILE=listings_large.csv \
  -e CALENDAR_FILE=calendar_large \
  spark jupyter lab ...
```

In [ ]:
import os
import socket
import time
import matplotlib.pyplot as plt
from IPython.display import display
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.storagelevel import StorageLevel

DATA_PATH = "/data"
timings = {}
total_start = time.perf_counter()

def force(df, label=""):
    """Persist + count to materialise a Spark DataFrame and measure true elapsed time."""
    df.persist(StorageLevel.MEMORY_AND_DISK)
    n = df.count()
    return df, n

print("Ready.")

In [ ]:
# Dataset / cluster selection — override with env vars for large data or cluster mode
LISTINGS_FILE = os.environ.get("LISTINGS_FILE", "listings.csv")
CALENDAR_FILE = os.environ.get("CALENDAR_FILE", "calendar.csv.gz")
SPARK_MASTER  = os.environ.get("SPARK_MASTER_URL", "local[*]")
print(f"Listings : {DATA_PATH}/{LISTINGS_FILE}")
print(f"Calendar : {DATA_PATH}/{CALENDAR_FILE}")
print(f"Master   : {SPARK_MASTER}")

## 1. Initialise Spark Session

Default: **local mode** — uses all CPU cores on this machine. Set `SPARK_MASTER_URL=spark://spark-master:7077` to connect to a cluster.

In [ ]:
t = time.perf_counter()
_builder = (
    SparkSession.builder
    .appName("AirbnbBenchmark")
    .master(SPARK_MASTER)
    .config("spark.driver.memory", "1g")
    .config("spark.sql.shuffle.partitions", "8")
)
if not SPARK_MASTER.startswith("local"):
    _builder = (
        _builder
        .config("spark.driver.host", socket.gethostbyname(socket.gethostname()))
        .config("spark.driver.bindAddress", "0.0.0.0")
        .config("spark.executor.memory", "512m")
    )
spark = _builder.getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
timings["spark_init"] = time.perf_counter() - t

print(f"Spark {spark.version} started  ({timings['spark_init']:.3f}s)")
print(f"Master  : {SPARK_MASTER}")
print(f"Cores   : {spark.sparkContext.defaultParallelism}")

## 2. Load Data

`multiLine=True` is required for `listings.csv` (description/amenities fields contain embedded newlines).

When `CALENDAR_FILE` is a **directory** (e.g. `calendar_large/`), Spark reads all part files in parallel — one partition per file. This is the key to Spark outperforming Pandas on larger data.

In [ ]:
t = time.perf_counter()
listings_raw = spark.read.option("multiLine", "true").csv(
    f"{DATA_PATH}/{LISTINGS_FILE}",
    header=True, inferSchema=False, quote='"', escape='"'
)
calendar_raw = spark.read.csv(
    f"{DATA_PATH}/{CALENDAR_FILE}",   # file path or directory of part files
    header=True, inferSchema=False
)
listings_raw, n_listings = force(listings_raw)
calendar_raw, n_calendar = force(calendar_raw)
timings["load_data"] = time.perf_counter() - t

print(f"Listings : {n_listings:,} rows × {len(listings_raw.columns)} cols  ({timings['load_data']:.3f}s)")
print(f"Calendar : {n_calendar:,} rows × {len(calendar_raw.columns)} cols")
display(listings_raw.select("id","neighbourhood_cleansed","room_type","price","number_of_reviews","review_scores_rating").limit(3).toPandas())

## 3. Clean Listings

Select columns, cast `id` to long, strip `$`/`,` from `price` with a regex, drop nulls.

In [ ]:
t = time.perf_counter()
listings = (
    listings_raw
    .select(
        F.col("id").cast("long").alias("listing_id"),
        F.col("neighbourhood_cleansed"),
        F.col("room_type"),
        F.regexp_replace(F.col("price"), r"[\$,]", "").cast("double").alias("price"),
        F.col("number_of_reviews").cast("int"),
        F.col("review_scores_rating").cast("double"),
        F.col("accommodates").cast("int"),
    )
    .filter(F.col("price").isNotNull() & (F.col("price") > 0))
)
listings, n = force(listings)
timings["clean_listings"] = time.perf_counter() - t

print(f"Cleaned listings: {n:,} rows  ({timings['clean_listings']:.3f}s)")
display(listings.limit(3).toPandas())

## 4. Clean Calendar

Convert `available` (t/f) to binary integer, parse `price`.

In [ ]:
t = time.perf_counter()
calendar = (
    calendar_raw
    .select(
        F.col("listing_id").cast("long"),
        F.col("date"),
        F.when(F.col("available") == "t", 1).otherwise(0).alias("is_available"),
        F.regexp_replace(F.col("price"), r"[\$,]", "").cast("double").alias("price"),
    )
)
calendar, n = force(calendar)
timings["clean_calendar"] = time.perf_counter() - t

print(f"Cleaned calendar: {n:,} rows  ({timings['clean_calendar']:.3f}s)")
display(calendar.limit(3).toPandas())

## 5. Aggregate Calendar Stats per Listing

For each listing compute total days, days available, avg booked price, and occupancy rate.

In [ ]:
t = time.perf_counter()
cal_agg = (
    calendar
    .groupBy("listing_id")
    .agg(
        F.count("*").alias("total_days"),
        F.sum("is_available").alias("days_available"),
        F.avg(
            F.when(F.col("is_available") == 0, F.col("price"))
        ).alias("avg_booked_price"),
    )
    .withColumn(
        "occupancy_rate",
        (F.col("total_days") - F.col("days_available")) / F.col("total_days"),
    )
)
cal_agg, n = force(cal_agg)
timings["calendar_agg"] = time.perf_counter() - t

print(f"Calendar aggregates: {n:,} listings  ({timings['calendar_agg']:.3f}s)")
display(cal_agg.limit(3).toPandas())

## 6. Join Listings with Calendar Aggregates

Left-join on `listing_id` so listings without calendar data are retained.

In [ ]:
t = time.perf_counter()
joined = listings.join(cal_agg, on="listing_id", how="left")
joined, n = force(joined)
timings["join"] = time.perf_counter() - t

print(f"Joined: {n:,} rows × {len(joined.columns)} cols  ({timings['join']:.3f}s)")
display(joined.select("listing_id","neighbourhood_cleansed","room_type","price","occupancy_rate","avg_booked_price").limit(3).toPandas())

## 7. Neighbourhood × Room Type Analysis

Group by `neighbourhood_cleansed` + `room_type`, sort by listing count descending.

In [ ]:
t = time.perf_counter()
neighbourhood_stats = (
    joined
    .groupBy("neighbourhood_cleansed", "room_type")
    .agg(
        F.count("*").alias("listing_count"),
        F.avg("price").alias("avg_price"),
        F.avg("occupancy_rate").alias("avg_occupancy"),
        F.avg("review_scores_rating").alias("avg_rating"),
    )
    .orderBy(F.desc("listing_count"))
)
top20 = neighbourhood_stats.limit(20).toPandas()
timings["neighbourhood_agg"] = time.perf_counter() - t

top20_display = top20.copy()
top20_display["avg_price"]     = top20_display["avg_price"].round(2)
top20_display["avg_occupancy"] = (top20_display["avg_occupancy"] * 100).round(1).astype(str) + "%"
top20_display["avg_rating"]    = top20_display["avg_rating"].round(2)
top20_display.columns = ["Neighbourhood", "Room Type", "Count", "Avg Price (€)", "Occupancy", "Rating"]
print(f"Computed in {timings['neighbourhood_agg']:.3f}s")
display(top20_display.style.background_gradient(subset=["Count"], cmap="Oranges").format(precision=2))

## 8. Room Type Summary

In [ ]:
t = time.perf_counter()
room_type_stats = (
    joined
    .groupBy("room_type")
    .agg(
        F.count("*").alias("count"),
        F.avg("price").alias("avg_price"),
        F.avg("occupancy_rate").alias("avg_occupancy"),
        F.avg("review_scores_rating").alias("avg_rating"),
    )
    .orderBy(F.desc("count"))
)
room_types = room_type_stats.toPandas()
timings["room_type_agg"] = time.perf_counter() - t
timings["total"] = time.perf_counter() - total_start

import pandas as _pd
display_rt = room_types.copy()
display_rt["avg_price"]     = display_rt["avg_price"].round(2)
display_rt["avg_occupancy"] = (display_rt["avg_occupancy"] * 100).round(1).astype(str) + "%"
display_rt["avg_rating"]    = display_rt["avg_rating"].round(2)
display_rt.columns = ["Room Type", "Count", "Avg Price (€)", "Occupancy", "Rating"]
display(display_rt.style.bar(subset=["Count"], color="#E07B39", width=80))

## 9. Timing Summary

In [ ]:
steps = {k: v for k, v in timings.items() if k != "total"}

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.barh(list(steps.keys()), list(steps.values()), color="#E07B39", edgecolor="white")
ax.bar_label(bars, fmt="%.3fs", padding=5, fontsize=10)
ax.set_xlabel("Seconds")
ax.set_title(f"Spark — Step Timing  (total: {timings['total']:.3f}s)", fontsize=13, fontweight="bold")
ax.invert_yaxis()
ax.set_xlim(0, max(steps.values()) * 1.25)
plt.tight_layout()
plt.show()

print(f"\nDataset : {LISTINGS_FILE}  |  {CALENDAR_FILE}")
print(f"Master  : {SPARK_MASTER}")
print(f"Total wall-clock time: {timings['total']:.3f}s")

In [ ]:
spark.stop()